In [1]:
# --- CÉLULA 1: Configuração e Importações ---
import sys
import os

# Adiciona a raiz do projeto ao path para acessar a pasta src/
sys.path.append(os.path.abspath(os.path.join('..')))

import torch
import torch.nn as nn
from torch_geometric.explain import Explainer, GNNExplainer

# Importando os módulos do nosso framework refatorado
from src.generators import SyntheticGraphGenerator
from src.models import GCNClassifier
from src.evaluator import Evaluator
from src.utils import split_train_test, train_gcn_model, visualizar_grafo_gt

# Garantir que o PyTorch use a GPU se disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Framework carregado com sucesso! Usando dispositivo: {device}")

/home/paulo/UFOP/ic/IC-Explainability_of_GNNs/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Framework carregado com sucesso! Usando dispositivo: cpu


In [2]:
torch.manual_seed(42)

print("1. Gerando Dados...")
generator_house = SyntheticGraphGenerator(
    num_nodes=300,
    num_houses=20,
    motif_type='house',
    feature_bias=1.0    # 2.0=fácil, 1.0=moderado, 0.5=difícil
)
data_house = generator_house.generate()
data_house = split_train_test(data_house, test_size=0.2).to(device)

print(f"   Nós: {data_house.num_nodes} | Classe 0: {(data_house.y==0).sum().item()} | Classe 1: {(data_house.y==1).sum().item()}")

print("\n2. Configurando Modelo...")
model_house = GCNClassifier(
    num_features=data_house.num_features, num_classes=2, hidden_dim=64
).to(device)
optimizer = torch.optim.Adam(model_house.parameters(), lr=0.01, weight_decay=5e-4)

counts = torch.bincount(data_house.y.cpu()).float()
weights = (1.0 / counts) / (1.0 / counts).max()
loss_fn = torch.nn.NLLLoss(weight=weights.to(device))

print("\n3. Treinando...")
model_house, metricas_house = train_gcn_model(
    model_house, data_house, optimizer, loss_fn, epochs=150, eval_interval=100
)

1. Gerando Dados...
   Nós: 400 | Classe 0: 300 | Classe 1: 100

2. Configurando Modelo...

3. Treinando...
Época  | Loss    | Acc    | Prec   | Rec    | F1    
-------------------------------------------------------
0      | 0.6481  | 0.3375 | 0.2740 | 1.0000 | 0.4301
100    | 0.1937  | 0.9875 | 1.0000 | 0.9500 | 0.9744
-------------------------------------------------------
🏆 Melhor Teste -> Acc: 0.9875 | Prec: 1.0000 | Rec: 0.9500 | F1: 0.9744


In [3]:
# --- CÉLULA 3: GNNExplainer ---
from src.explainers import GNNExplainerWrapper

explainer = GNNExplainerWrapper(model_house, epochs=200)

# Explica todos os nós de motif e agrega as máscaras
all_masks, agg_mask = explainer.explain_all_motif_nodes(data_house)

print(f"\nMáscara agregada — min: {agg_mask.min():.4f} | max: {agg_mask.max():.4f} | média: {agg_mask.mean():.4f}")

Explicando 100 nós de motif...
  Nó 1/100...
  Nó 11/100...
  Nó 21/100...
  Nó 31/100...
  Nó 41/100...
  Nó 51/100...
  Nó 61/100...
  Nó 71/100...
  Nó 81/100...
  Nó 91/100...
Concluído! 100 explicações geradas.

Máscara agregada — min: 0.0000 | max: 1.0000 | média: 0.0893


In [4]:
# --- CÉLULA 4: Avaliação das Explicações ---
from src.evaluator import Evaluator

evaluator = Evaluator()
gt_mask = data_house.edge_mask_gt.cpu()

# --- Métricas agregadas (visão geral do explicador) ---
auroc = Evaluator.calculate_auroc(agg_mask, gt_mask)
jaccard = Evaluator.calculate_jaccard_accuracy(agg_mask, gt_mask)
recall = Evaluator.calculate_recall(agg_mask, gt_mask)

print("=== Métricas da Explicação Agregada ===")
print(f"  AUC-ROC  : {auroc:.4f}  (1.0 = perfeito, 0.5 = aleatório)")
print(f"  Jaccard  : {jaccard:.4f}  (1.0 = explicação idêntica ao GT)")
print(f"  Recall   : {recall:.4f}  (quanto do motif foi recuperado)")

# --- Fidelity por nó (avalia cada explicação individualmente) ---
fid_plus_list, fid_minus_list, unfaith_list = [], [], []
motif_indices = (data_house.y == 1).nonzero(as_tuple=True)[0].tolist()

for node_idx in motif_indices:
    edge_mask = all_masks[node_idx].to(device)
    fp, fm = Evaluator.calculate_fidelity(model_house, data_house, node_idx, edge_mask)
    uf = Evaluator.calculate_unfaithfulness(model_house, data_house, node_idx, edge_mask)
    fid_plus_list.append(fp)
    fid_minus_list.append(fm)
    unfaith_list.append(uf)

print("\n=== Fidelity (média sobre nós de motif) ===")
print(f"  Fidelity+     : {sum(fid_plus_list)/len(fid_plus_list):.4f}  (prob. mantendo só a explicação)")
print(f"  Fidelity-     : {sum(fid_minus_list)/len(fid_minus_list):.4f}  (queda de prob. removendo a explicação)")
print(f"  Unfaithfulness: {sum(unfaith_list)/len(unfaith_list):.4f}  (0.0 = fiel ao modelo)")

=== Métricas da Explicação Agregada ===
  AUC-ROC  : 0.9932  (1.0 = perfeito, 0.5 = aleatório)
  Jaccard  : 0.7597  (1.0 = explicação idêntica ao GT)
  Recall   : 0.8167  (quanto do motif foi recuperado)

=== Fidelity (média sobre nós de motif) ===
  Fidelity+     : 0.9436  (prob. mantendo só a explicação)
  Fidelity-     : 0.3874  (queda de prob. removendo a explicação)
  Unfaithfulness: 0.0919  (0.0 = fiel ao modelo)


In [5]:
# --- CÉLULA 5: Experimento Star ---
torch.manual_seed(42)

print("=== Experimento: Motif Star ===")
generator_star = SyntheticGraphGenerator(
    num_nodes=300, num_houses=20, motif_type='star', feature_bias=1.0
)
data_star = generator_star.generate()
data_star = split_train_test(data_star, test_size=0.2).to(device)

model_star = GCNClassifier(num_features=data_star.num_features, num_classes=2, hidden_dim=64).to(device)
optimizer_star = torch.optim.Adam(model_star.parameters(), lr=0.01, weight_decay=5e-4)

counts_star = torch.bincount(data_star.y.cpu()).float()
weights_star = (1.0 / counts_star) / (1.0 / counts_star).max()
loss_fn_star = torch.nn.NLLLoss(weight=weights_star.to(device))

model_star, metricas_star = train_gcn_model(
    model_star, data_star, optimizer_star, loss_fn_star, epochs=150, eval_interval=10
)

# GNNExplainer
explainer_star = GNNExplainerWrapper(model_star, epochs=200)
all_masks_star, agg_mask_star = explainer_star.explain_all_motif_nodes(data_star)

# Avaliação
gt_mask_star = data_star.edge_mask_gt.cpu()
auroc_star = Evaluator.calculate_auroc(agg_mask_star, gt_mask_star)
jaccard_star = Evaluator.calculate_jaccard_accuracy(agg_mask_star, gt_mask_star)
recall_star = Evaluator.calculate_recall(agg_mask_star, gt_mask_star)

fid_plus_star, fid_minus_star, unfaith_star = [], [], []
for node_idx in (data_star.y == 1).nonzero(as_tuple=True)[0].tolist():
    em = all_masks_star[node_idx].to(device)
    fp, fm = Evaluator.calculate_fidelity(model_star, data_star, node_idx, em)
    uf = Evaluator.calculate_unfaithfulness(model_star, data_star, node_idx, em)
    fid_plus_star.append(fp)
    fid_minus_star.append(fm)
    unfaith_star.append(uf)

print(f"\nStar — AUC: {auroc_star:.4f} | Jaccard: {jaccard_star:.4f} | Recall: {recall_star:.4f}")
print(f"       Fid+: {sum(fid_plus_star)/len(fid_plus_star):.4f} | Fid-: {sum(fid_minus_star)/len(fid_minus_star):.4f} | Unfaith: {sum(unfaith_star)/len(unfaith_star):.4f}")

=== Experimento: Motif Star ===
Época  | Loss    | Acc    | Prec   | Rec    | F1    
-------------------------------------------------------
0      | 0.6736  | 0.4048 | 0.3243 | 1.0000 | 0.4898
10     | 0.1817  | 0.9405 | 0.8800 | 0.9167 | 0.8980
20     | 0.0980  | 0.9405 | 0.8800 | 0.9167 | 0.8980
30     | 0.0952  | 0.9405 | 0.8800 | 0.9167 | 0.8980
40     | 0.1256  | 0.9405 | 0.8800 | 0.9167 | 0.8980
50     | 0.0898  | 0.9405 | 0.8800 | 0.9167 | 0.8980
60     | 0.1150  | 0.9405 | 0.8800 | 0.9167 | 0.8980
70     | 0.0897  | 0.9405 | 0.8800 | 0.9167 | 0.8980
80     | 0.0864  | 0.9405 | 0.8800 | 0.9167 | 0.8980
90     | 0.0878  | 0.9405 | 0.8800 | 0.9167 | 0.8980
100    | 0.0852  | 0.9405 | 0.8800 | 0.9167 | 0.8980
110    | 0.0893  | 0.9643 | 0.9565 | 0.9167 | 0.9362
120    | 0.0613  | 0.9405 | 0.8800 | 0.9167 | 0.8980
130    | 0.0781  | 0.9405 | 0.8800 | 0.9167 | 0.8980
140    | 0.0746  | 0.9405 | 0.8800 | 0.9167 | 0.8980
-------------------------------------------------------
🏆 Melhor